In [1]:
IS_ONLINE_JUDGE = False

# =================================
import time

GLOBAL_START_TIME = time.perf_counter()


def debug_print(*args, **kwargs):
    if IS_ONLINE_JUDGE:
        return
    print(*args, **kwargs)


def print_elapsed_time():
    debug_print(f"elapsed time: {(time.perf_counter() - GLOBAL_START_TIME) * 1000:.2f}msecs")


# =================================
import heapq
import math

INF = 10**18
ESTIMATE_MAX_TIME = 1.28

N_POINT = 80
N_PAIR = 40
CUT_RATIO = 0.05

FILE_NUM = 100
TARGET_LENGTH = 2000


# =================================


def calc_dist(p1: tuple, p2: tuple) -> float:
    return math.dist(p1, p2)


def prim(
    graph: dict[int, list[tuple[int, int]]],
) -> tuple[list[tuple[int, int]], list[int]]:
    """
    G: 隣接グラフ
    G := [ [(v_0, cost_0), (v_1,cost_1),..], [(v_2, cost_2)],...]

    返り値 ans :最小全域木の重みの総和
    """
    v_list = list(graph.keys())
    used = {v: False for v in v_list}

    init_v = v_list[0]  # 適当な点を選ぶ
    used[init_v] = True
    que = [(cost, init_v, v) for v, cost in graph[init_v]]
    heapq.heapify(que)

    ans_v: list[int] = [init_v]
    ans_edges: list[tuple[int, int]] = []
    while que:
        cost_v, from_v, to_v = heapq.heappop(que)
        if used[to_v]:
            continue
        used[to_v] = True
        ans_edges.append((min(from_v, to_v), max(from_v, to_v)))
        ans_v.append(to_v)
        for nxt, cost_nxt in graph[to_v]:
            if used[nxt]:
                continue
            heapq.heappush(que, (cost_nxt, to_v, nxt))
    return ans_edges, ans_v


def prim_k(graph: list[list[float]], init_v: int, k: int, used_v: set[int], N):
    """
    init_vを含むk頂点の最小全域木を求める
    """
    used = used_v | {init_v}
    used.add(init_v)
    que = [(graph[init_v][v], init_v, v) for v in range(N)]
    heapq.heapify(que)

    ans_v = [init_v]
    ans_edges: list[tuple[int, int]] = []
    ans_cost = 0.0
    while que and len(ans_v) < k:
        cost_v, from_v, to_v = heapq.heappop(que)
        if to_v in used:
            continue
        used.add(to_v)
        ans_edges.append((min(from_v, to_v), max(from_v, to_v)))
        ans_v.append(to_v)
        ans_cost += cost_v
        for nxt in range(N):
            if nxt in used:
                continue
            heapq.heappush(que, (graph[to_v][nxt], to_v, nxt))
    return ans_edges, ans_v, ans_cost


# =================================


class EnvOffline:
    def __init__(self, input_file_path: str, output_file_path: str):
        with open(input_file_path) as f:
            lines = f.readlines()
            N, M, Q, L, W = map(int, lines[0].split())
            G = list(map(int, lines[1].split()))
            rectangles = []
            for l in range(2, 2 + N):
                rectangles.append(list(map(int, lines[l].split())))
            coordinates = []
            for l in range(2 + N, 2 + N + N):
                coordinates.append(tuple(map(int, lines[l].split())))

        self.N, self.M, self.Q, self.L, self.W = N, M, Q, L, W
        self.G = G
        self.rectangles = rectangles
        self.coordinates = coordinates
        self.cneter_points = [((l + r) / 2, (c + d) / 2) for l, r, c, d in rectangles]
        self._query_history: list[str] = []

        self.input_file_path = input_file_path
        self.output_file_path = output_file_path

    def query(self, c_list: list[int]) -> list[tuple[int, int]]:
        query_str = " ".join(["?", str(len(c_list)), *map(str, c_list)])
        self._query_history.append(query_str)

        now_graph: dict[int, list] = {v: [] for v in c_list}
        for c1 in c_list:
            for c2 in c_list:
                if c1 == c2:
                    continue
                x1, y1 = self.coordinates[c1]
                x2, y2 = self.coordinates[c2]
                dist = abs(x1 - x2) ** 2 + abs(y1 - y2) ** 2
                now_graph[c1].append((c2, dist))
                now_graph[c2].append((c1, dist))

        ans_edges, ans_v = prim(now_graph)
        return ans_edges

    def answer(
        self,
        groups: list[list[int]],
        edges: list[list[tuple[int, int]]],
    ):
        cost = 0.0
        ans_str = "!\n"
        for i in range(len(groups)):
            ans_str += " ".join(map(str, groups[i])) + "\n"
            for e in edges[i]:
                ans_str += " ".join(map(str, e)) + "\n"
                dist = calc_dist(self.coordinates[e[0]], self.coordinates[e[1]])
                cost += dist

        for query_str in self._query_history:
            ans_str += query_str + "\n"

        with open(self.output_file_path, "w") as f:
            f.write(ans_str)

        return cost


def solve(env: EnvOffline, target_points):
    graph: list[list[float]] = [[0 for _ in range(env.N)] for _ in range(env.N)]
    for i in range(env.N):
        for j in range(env.N):
            if i == j:
                continue
            dist = calc_dist(target_points[i], target_points[j])
            graph[i][j] = dist
            graph[j][i] = dist

    groups = [(i, g) for i, g in enumerate(env.G)]
    groups = sorted(groups, key=lambda x: x[1], reverse=True)

    ans_edges = [None for _ in range(len(groups))]
    ans_v = [None for _ in range(len(groups))]

    now_used: set = set()
    not_used: set = set(range(env.N))
    for group_n, group_size in groups:
        v = not_used.pop()
        prim_edges, prim_v, _ = prim_k(graph, v, group_size, now_used, env.N)

        ans_edges[group_n] = prim_edges
        ans_v[group_n] = prim_v
        now_used.update(prim_v)
        not_used.difference_update(prim_v)

    return ans_v, ans_edges


def estimate_point_cost(points1, points2):
    all_dist = []
    for n in range(len(points1)):
        all_dist.append(calc_dist(points1[n], points2[n]))
    total = sum(all_dist)
    ave = sum(all_dist) / len(all_dist)
    return ave, total

In [ ]:
costs = []
for file_num in range(FILE_NUM):
    debug_print("=====")
    start_time = time.perf_counter()
    input_file_path = f"../in/{file_num:04d}.txt"
    output_file_path = f"../out/{file_num:04d}.txt"
    env = EnvOffline(input_file_path, output_file_path)
    ans_v, ans_edges = solve(env, env.coordinates)
    cost = env.answer(ans_v, ans_edges)
    debug_print(f"[{file_num}/{FILE_NUM}]: {cost} {time.perf_counter() - start_time:.2f}s")
    costs.append(cost)
avg = sum(costs) / len(costs)
debug_print(f"avg: {avg}")
with open("../result/base.txt", "w") as f:
    f.write(f"avg: {avg}\n")
    for pi, cost in enumerate(costs):
        f.write(f"{pi:0>4} {cost}\n")

=====
[0/100]: 177088.33648208648 0.54s
=====
[1/100]: 203407.2242962117 0.33s
=====
[2/100]: 200986.30285379454 0.50s
=====
[3/100]: 156914.0862541892 0.29s
=====
[4/100]: 171290.2188162318 0.30s
=====
[5/100]: 157974.65995222153 0.27s
=====
[6/100]: 187303.05891821499 0.43s
=====
[7/100]: 200144.35641554144 0.39s
=====
[8/100]: 208993.3890733209 0.32s
=====
[9/100]: 194272.64625778524 0.26s
=====
[10/100]: 197439.1675193934 0.29s
=====
[11/100]: 195458.34051951897 0.30s
=====
[12/100]: 191018.73072793422 0.28s
=====
[13/100]: 161342.95695367403 0.22s
=====
[14/100]: 152720.1510536726 0.26s
=====
[15/100]: 130233.56020157982 0.22s
=====
[16/100]: 195007.7094522924 0.24s
=====
[17/100]: 201679.17930320973 0.21s
=====
[18/100]: 194256.5924717526 0.24s
=====
[19/100]: 92679.81061850977 0.21s
=====
[20/100]: 197073.84242743175 0.24s
=====
[21/100]: 202365.48583155946 0.21s
=====
[22/100]: 209774.8324134294 0.24s
=====
[23/100]: 206233.5418667354 0.23s
=====
[24/100]: 173076.78717937393 0.